# Paper 1 backbone on YOLO26n: ADown down-sampling

| Part | Source |
|---|---|
| **Backbone (layers 0–10)** | **Paper 1:** the stride-2 `Conv` at layers 3, 5, 7 replaced by **ADown** |
| Neck FPN + PAN (layers 11–22) | YOLO26n base, unchanged |
| Head (Detect, layer 23) | YOLO26n base, unchanged (one-to-one, NMS-free) |

The baseline ("Standard YOLO26n Backbone") is **not trained here**: it is the same model as the baseline of the Paper 7 notebook,
so its values are already filled in the final table.

**Kaggle settings:** Accelerator = **GPU T4 x1**, Internet = **On**

## 1. Setup

In [1]:
# ultralytics version is pinned: use the same version on Kaggle, Raspberry Pi 5 and Jetson
!pip install -q ultralytics==8.4.137 roboflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.9/373.9 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 112.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 5.5 MB/s eta 0:00:00


In [2]:
import os, sys, gc, shutil, random
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import torch, torchvision
import ultralytics

print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| torchvision", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (turn on the GPU accelerator)")

WORK = Path("/kaggle/working")
os.chdir(WORK)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
ultralytics 8.4.137 | torch 2.11.0+cu128 | torchvision 0.26.0+cu128
GPU: Tesla T4


## 2. Dataset (download from Roboflow)

The Roboflow API key is written in the cell below. Keep this notebook **Private** on Kaggle.

In [3]:
import os, yaml
from roboflow import Roboflow

os.chdir("/kaggle/working")

# Roboflow API key (keep this notebook Private on Kaggle)
API_KEY = "CxHgc1Nn3FK5vYNsBtJW"

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("md-mahibullah").project("digitalaccess-2-copy")
dataset = project.version(4).download("yolo26")  # Roboflow YOLO26 export (same label format as YOLOv8)

# Rename and fix paths
new = "/kaggle/working/digitalaccess"
if os.path.exists(dataset.location) and not os.path.exists(new):
    os.rename(dataset.location, new)

# Update YAML
yaml_path = f"{new}/data.yaml"
with open(yaml_path) as f:
    data = yaml.safe_load(f)
data["path"], data["train"], data["val"] = new, "train/images", "valid/images"
if os.path.exists(f"{new}/test/images"):
    data["test"] = "test/images"
else:
    data.pop("test", None)
with open(yaml_path, "w") as f:
    yaml.dump(data, f)

# the following cells use these variables
DATA_ROOT = Path(new)
DATA_YAML = yaml_path
names = data["names"]
assert data.get("test"), "No test split: the evaluation and FPS steps need a test split"

for s in ["train", "valid", "test"]:
    print(f"{s:6s}: {len(list((DATA_ROOT / s / 'images').glob('*')))} images")
print("nc =", len(names))
print("✅ Dataset prepared!")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to digitalaccess-2(copy)-4 in yolo26:: 100%|██████████| 17613/17613 [00:01<00:00, 11297.35it/s]


train : 7704 images
valid : 550 images
test  : 550 images
nc = 27
✅ Dataset prepared!


## 3. Paper 1 backbone: ADown

**What ADown does.** A 2×2 average pool (stride 1) smooths the input first, which reduces aliasing when the resolution is halved.
The channels are then split into two halves:
one half goes through a 3×3 stride-2 conv, the other through a 3×3 stride-2 max-pool and a 1×1 conv; the two halves are concatenated.

**Where it sits in YOLO26n.**

| Layer | Base YOLO26n | Paper 1 backbone | Output |
|---|---|---|---|
| 3 | Conv 3×3, stride 2 | **ADown** | P3/8 (80 × 80) |
| 5 | Conv 3×3, stride 2 | **ADown** | P4/16 (40 × 40) |
| 7 | Conv 3×3, stride 2 | **ADown** | P5/32 (20 × 20) |

One Conv is replaced by one ADown, so no layer index changes and the neck connections stay the same.
Only layers 3, 5 and 7 start from scratch; every other layer (rest of the backbone, the whole neck and the head) loads `yolo26n.pt`.

**Why it fits the task.** ADown has fewer parameters and FLOPs than the 3×3 stride-2 conv it replaces, which raises the speed on a
Raspberry Pi 5, and it is a built-in ultralytics module, so it exports to NCNN with standard layers and needs no extra file on the devices.

**Why the other Paper 1 modules are not used.**
- *ECA* does not make the model lighter, and YOLO26n already has C2PSA attention.
- *CG-C3k2* targets small objects in clutter; the subjects here are large and close to the camera.
- *P2 branch, P2 head, LSCD, 4-head TaskAligned Assigner* (neck/head) are designed for very small objects and slow the model down on a Raspberry Pi 5; the neck and head stay YOLO26n base here.

In [4]:
%%writefile yolo26n-bb-adown.yaml
# YOLO26n with the Paper 1 backbone: stride-2 Conv at layers 3, 5, 7 -> ADown. Neck and head = YOLO26n base.
nc: 80 # overridden by the dataset
end2end: True
reg_max: 1
scales:
  n: [0.50, 0.25, 1024]

backbone:
  - [-1, 1, Conv, [64, 3, 2]] # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]] # 1-P2/4
  - [-1, 2, C3k2, [256, False, 0.25]]
  - [-1, 1, ADown, [256]] # 3-P3/8   (Conv -> ADown)
  - [-1, 2, C3k2, [512, False, 0.25]]
  - [-1, 1, ADown, [512]] # 5-P4/16  (Conv -> ADown)
  - [-1, 2, C3k2, [512, True]]
  - [-1, 1, ADown, [1024]] # 7-P5/32 (Conv -> ADown)
  - [-1, 2, C3k2, [1024, True]]
  - [-1, 1, SPPF, [1024, 5, 3, True]] # 9
  - [-1, 2, C2PSA, [1024]] # 10

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]] # cat backbone P4
  - [-1, 2, C3k2, [512, True]] # 13

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]] # cat backbone P3
  - [-1, 2, C3k2, [256, True]] # 16 (P3/8-small)

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]] # cat head P4
  - [-1, 2, C3k2, [512, True]] # 19 (P4/16-medium)

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 10], 1, Concat, [1]] # cat head P5
  - [-1, 1, C3k2, [1024, True, 0.5, True]] # 22 (P5/32-large)

  - [[16, 19, 22], 1, Detect, [nc]] # Detect(P3, P4, P5)

Writing yolo26n-bb-adown.yaml


In [5]:
from ultralytics.nn.tasks import DetectionModel, yaml_model_load
from ultralytics.utils.torch_utils import get_flops

MODEL_YAML = str(WORK / "yolo26n-bb-adown.yaml")

def fused_params_gflops(model, imgsz=640):
    m = deepcopy(model).float().cpu().eval()
    m.fuse(verbose=False)
    return sum(p.numel() for p in m.parameters()) / 1e6, get_flops(m, imgsz)

# structure check (untrained): base vs Paper 1 backbone
base = DetectionModel(yaml_model_load("yolo26n.yaml"), nc=len(names), verbose=False)
adown = DetectionModel(yaml_model_load(MODEL_YAML), nc=len(names), verbose=False)
adown.eval(); _ = adown(torch.zeros(1, 3, 640, 640))  # forward check
print("layers 3, 5, 7:", [type(adown.model[i]).__name__ for i in (3, 5, 7)])
for tag, m in (("base", base), ("Paper 1 backbone", adown)):
    pm, gf = fused_params_gflops(m)
    print(f"{tag:17s} Params {pm:.3f} M | GFLOPs {gf:.2f}")

Overriding model.yaml nc=80 with nc=27
Overriding model.yaml nc=80 with nc=27
layers 3, 5, 7: ['ADown', 'ADown', 'ADown']
base              Params 2.380 M | GFLOPs 5.34
Paper 1 backbone  Params 2.034 M | GFLOPs 4.49


## 4. Training

Same settings as all earlier notebooks (and the baseline). Only the Paper 1 backbone model is trained.

In [6]:
from ultralytics import YOLO

EPOCHS  = 100
IMGSZ   = 640
BATCH   = 16
WORKERS = 2
SEED    = 0
PROJECT = str(WORK / "runs")

COMMON = dict(
    data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=SEED, deterministic=True,
    pretrained="yolo26n.pt", device=0, workers=WORKERS, cache=False, patience=30, cos_lr=False,
    project=PROJECT, exist_ok=True, plots=True,
    # channels_last=False: ultralytics' MuSGD crashes on channels_last weights ("view size is not compatible").
    optimizer="MuSGD", lr0=0.01, momentum=0.9, channels_last=False,
)

# (row label in the table, run name)
VARIANTS = [
    ("ADown Backbone (Paper 1)", "bb_adown"),
]

def best_path(run):
    return Path(PROJECT) / f"{run}_s{SEED}" / "weights" / "best.pt"

for label, run in VARIANTS:
    print(f"\n========== {label} ({run}) ==========")
    model = YOLO(MODEL_YAML)
    model.train(**dict(COMMON, name=f"{run}_s{SEED}"))
    del model; gc.collect(); torch.cuda.empty_cache()


========== ADown Backbone (Paper 1) (bb_adown) ==========
New https://pypi.org/project/ultralytics/8.4.183 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.137 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/digitalaccess/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None

`get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.


      2/100      3.36G      1.394      3.171    0.01911         14        640: 100% ━━━━━━━━━━━━ 482/482 4.7it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 7.3it/s 2.5s
                   all        550        557       0.72      0.682      0.792       0.41

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      3/100      3.36G      1.422      1.869    0.01941         12        640: 100% ━━━━━━━━━━━━ 482/482 4.8it/s 1:39
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 7.4it/s 2.4s
                   all        550        557      0.742        0.7      0.837      0.483

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      4/100      3.36G      1.441      1.465    0.01967          7        640: 100% ━━━━━━━━━━━━ 482/482 4.9it/s 1:39
                 Class     Images  Instances      Box(P 

## 5. Test-set evaluation: Precision, Recall, F1-score, mAP@50, Params, GFLOPs

Params and GFLOPs are counted on the fused inference model, the same way as the baseline row.

In [7]:
rows = []
for label, run in VARIANTS:
    w = best_path(run)
    if not w.exists():
        print(f"{run}: best.pt not found, train it first"); continue
    r = YOLO(str(w)).val(data=DATA_YAML, split="test", imgsz=IMGSZ, batch=16, device=0, plots=False, verbose=False)
    P, R = float(r.box.mp), float(r.box.mr)
    params, gflops = fused_params_gflops(YOLO(str(w)).model, IMGSZ)
    rows.append({"run": run, "Backbone Variant": label,
                 "Params (M)": round(params, 2), "GFLOPs": round(gflops, 2),
                 "Precision": round(P, 3), "Recall": round(R, 3),
                 "F1-score": round(2 * P * R / (P + R + 1e-9), 3),
                 "mAP@50": round(float(r.box.map50), 3)})
METRICS = pd.DataFrame(rows)
METRICS.drop(columns="run")

Ultralytics 8.4.137 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO26n-bb-adown summary (fused): 125 layers, 2,033,989 parameters, 0 gradients, 4.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1129.0±287.8 MB/s, size: 37.7 KB)
val: Scanning /kaggle/working/digitalaccess/test/labels... 550 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 550/550 1.3Kit/s 0.4s
val: New cache created: /kaggle/working/digitalaccess/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 35/35 13.4it/s 2.6s
                   all        550        552      0.967      0.925      0.952      0.581
Speed: 0.5ms preprocess, 2.5ms inference, 0.0ms loss, 0.1ms postprocess per image


,Backbone Variant,Params (M),GFLOPs,Precision,Recall,F1-score,mAP@50
0,ADown Backbone (Paper 1),2.03,4.49,0.967,0.925,0.946,0.952


## 6. Package for Raspberry Pi 5 and Jetson

Creates `paper1_backbone_edge_bench.zip` in the **Output** tab with:

| Item | Used on |
|---|---|
| `bb_adown_ncnn_model/` (NCNN model, imgsz 640) | Raspberry Pi 5 |
| `bb_adown.pt` | Jetson (ADown is built into ultralytics, no extra file needed) |
| `bench_images/` (the same 50 real test images as the earlier packages, seed 0) | both |
| `pi5_bench.py`, `jetson_bench.py` | benchmark scripts |

Both scripts: batch 1, imgsz 640, cycle through the 50 real test images (loaded into memory first),
100 warm-up runs, then 1000 timed runs end-to-end (preprocessing + inference + postprocessing), and print the **median** ms and FPS.
Unzip into its own folder, e.g. `python -m zipfile -e paper1_backbone_edge_bench.zip paper1_backbone_edge_bench`.

In [8]:
EDGE_IMGSZ = 640
N_BENCH_IMAGES = 50
EDGE = WORK / "paper1_backbone_edge_bench"
shutil.rmtree(EDGE, ignore_errors=True)
EDGE.mkdir()

# the same 50 real test images as the earlier packages (same seed and selection)
test_imgs = sorted(p for p in (DATA_ROOT / "test" / "images").glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"})
random.Random(SEED).shuffle(test_imgs)
(EDGE / "bench_images").mkdir()
for p in test_imgs[:N_BENCH_IMAGES]:
    shutil.copy(p, EDGE / "bench_images" / p.name)
print(f"bench_images: {len(list((EDGE / 'bench_images').glob('*')))} real test images")

for _, run in VARIANTS:
    w = best_path(run)
    if not w.exists():
        print(f"{run}: best.pt not found, skipped"); continue
    shutil.copy(w, EDGE / f"{run}.pt")
    p = YOLO(str(w)).export(format="ncnn", imgsz=EDGE_IMGSZ, half=False)
    shutil.copytree(p, EDGE / f"{run}_ncnn_model")  # ultralytics needs the "_ncnn_model" suffix
    print(f"{run:12s} -> {run}.pt, {run}_ncnn_model/")

COMMON_BENCH = """
import glob, statistics, sys, time
import cv2

IMGSZ, WARMUP, RUNS = {imgsz}, 100, 1000
imgs = [cv2.imread(p) for p in sorted(glob.glob("bench_images/*"))]
imgs = [im for im in imgs if im is not None]
if not imgs:
    sys.exit("No images found in bench_images/. Run this script inside the unzipped folder.")
print(f"{{len(imgs)}} real test images, imgsz {{IMGSZ}}, batch 1, {{WARMUP}} warm-up + {{RUNS}} timed runs (median)")


def bench(model, **kw):
    for i in range(WARMUP):
        model.predict(imgs[i % len(imgs)], imgsz=IMGSZ, verbose=False, **kw)
    times = []
    for i in range(RUNS):
        t = time.perf_counter()
        model.predict(imgs[i % len(imgs)], imgsz=IMGSZ, verbose=False, **kw)
        times.append((time.perf_counter() - t) * 1000)
    return statistics.median(times)
""".format(imgsz=EDGE_IMGSZ)

PI5 = '''# Raspberry Pi 5 benchmark (NCNN). Run inside the unzipped folder:  python pi5_bench.py
from ultralytics import YOLO
''' + COMMON_BENCH + '''
models = sorted(glob.glob("*_ncnn_model"))
if not models:
    sys.exit("No *_ncnn_model folders found.")
for path in models:
    ms = bench(YOLO(path, task="detect"))
    print(f"{path.replace('_ncnn_model', ''):12s}  median {ms:8.2f} ms   {1000 / ms:7.2f} FPS")
'''

JETSON = '''# Jetson benchmark (PyTorch, CUDA FP16). Run inside the unzipped folder:  python jetson_bench.py
import sys
import torch

if not torch.cuda.is_available():
    sys.exit("WARNING: CUDA GPU not available. Install the NVIDIA Jetson PyTorch wheel for your JetPack, then run again.")
print("GPU:", torch.cuda.get_device_name(0))

from ultralytics import YOLO
''' + COMMON_BENCH + '''
models = sorted(glob.glob("*.pt"))
if not models:
    sys.exit("No .pt files found.")
for path in models:
    ms = bench(YOLO(path, task="detect"), device=0, half=True)
    print(f"{path.replace('.pt', ''):12s}  median {ms:8.2f} ms   {1000 / ms:7.2f} FPS")
'''

(EDGE / "pi5_bench.py").write_text(PI5)
(EDGE / "jetson_bench.py").write_text(JETSON)
shutil.make_archive(str(WORK / "paper1_backbone_edge_bench"), "zip", EDGE)
print("\nDownload paper1_backbone_edge_bench.zip from the Output tab.")

bench_images: 50 real test images
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.137 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
WARNING ⚠️ This export format does not support end2end models, disabling the end2end branch.
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO26n-bb-adown summary (fused): 149 layers, 2,158,722 parameters, 0 gradients, 4.5 GFLOPs

PyTorch: starting from '/kaggle/working/runs/bb_adown_s0/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 31, 8400) (4.5 MB)
requirements: Ultralytics requirement ['ncnn'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 1 package in 175ms
 Downloaded ncnn
Prepared 1 package in 168ms
Installed 1 package in 2ms
 + ncnn==1.0.20261020

requirements: AutoUpdate success ✅ 0.8s
WARNING ⚠️ requir

pnnxparam = /kaggle/working/runs/bb_adown_s0/weights/best_ncnn_model/model.pnnx.param
pnnxbin = /kaggle/working/runs/bb_adown_s0/weights/best_ncnn_model/model.pnnx.bin
pnnxpy = /kaggle/working/runs/bb_adown_s0/weights/best_ncnn_model/model_pnnx.py
pnnxonnx = /kaggle/working/runs/bb_adown_s0/weights/best_ncnn_model/model.pnnx.onnx
ncnnparam = /kaggle/working/runs/bb_adown_s0/weights/best_ncnn_model/model.ncnn.param
ncnnbin = /kaggle/working/runs/bb_adown_s0/weights/best_ncnn_model/model.ncnn.bin
ncnnpy = /kaggle/working/runs/bb_adown_s0/weights/best_ncnn_model/model_ncnn.py
fp16 = 0
optlevel = 2
device = cpu
inputshape = [1,3,640,640]f32
inputshape2 = 
input = 
input2 = 
customop = 
moduleop = 
get inputshape from traced inputs
inputshape = [1,3,640,640]f32
############# pass_level0
inline module = torch.nn.modules.linear.Identity
inline module = ultralytics.nn.modules.block.ADown
inline module = ultralytics.nn.modules.block.Attention
inline module = ultralytics.nn.modules.block.Bottlen

NCNN: export success ✅ 8.5s, saved as '/kaggle/working/runs/bb_adown_s0/weights/best_ncnn_model' (8.0 MB)

Export complete (9.0s)
Results saved to /kaggle/working/runs/bb_adown_s0/weights/best_ncnn_model
Predict:         yolo predict task=detect model=/kaggle/working/runs/bb_adown_s0/weights/best_ncnn_model imgsz=640 
Validate:        yolo val task=detect model=/kaggle/working/runs/bb_adown_s0/weights/best_ncnn_model imgsz=640 data=/kaggle/working/digitalaccess/data.yaml  
Visualize:       https://netron.app
bb_adown     -> bb_adown.pt, bb_adown_ncnn_model/

Download paper1_backbone_edge_bench.zip from the Output tab.


### How to run on the devices

**Raspberry Pi 5** (same Pi and same conditions as the baseline measurement)
1. Activate the Python environment that has `ultralytics==8.4.137` and `ncnn`.
2. `echo performance | sudo tee /sys/devices/system/cpu/cpu*/scaling_governor`
3. Go into the unzipped folder and run `python pi5_bench.py`.
4. Run it 3 times with a 2–3 minute pause; use the middle value. Check `vcgencmd measure_temp` and `vcgencmd get_throttled` (`throttled=0x0` = OK).

**Jetson**
1. NVIDIA Jetson PyTorch + torchvision for your JetPack, then `ultralytics==8.4.137`.
2. `sudo nvpmodel -m 0` (check with `sudo nvpmodel -q`) and `sudo jetson_clocks`.
3. Go into the unzipped folder and run `python jetson_bench.py`. Run 3 times; use the middle value.

## 7. Final backbone table

The baseline row is filled with the values from the Paper 7 notebook (same model). Type the median FPS of the Paper 1 backbone model
(and the baseline Jetson FPS when you have it), keep `None` for a missing value (shown as `--`), then run the cell.
Run section 5 first in the same session (it fills `METRICS`).

In [9]:
# baseline values from the Paper 7 notebook (identical YOLO26n model)
BASE_ROW = {"Backbone Variant": "Standard YOLO26n Backbone", "Params (M)": 2.38, "GFLOPs": 5.34,
            "Precision": 0.971, "Recall": 0.937, "F1-score": 0.953, "mAP@50": 0.958,
            "FPS (Rasp-5)": 13.38, "FPS (Jetson)": None}

FPS_PI5    = {"bb_adown": None}
FPS_JETSON = {"bb_adown": None}

table = METRICS.copy()
table["FPS (Rasp-5)"] = table["run"].map(lambda r: FPS_PI5.get(r))
table["FPS (Jetson)"] = table["run"].map(lambda r: FPS_JETSON.get(r))
COLS = ["Backbone Variant", "Params (M)", "GFLOPs", "Precision", "Recall", "F1-score", "mAP@50", "FPS (Rasp-5)", "FPS (Jetson)"]
table = pd.concat([pd.DataFrame([BASE_ROW]), table[COLS]], ignore_index=True)[COLS]
table = table.astype(object).where(table.notna(), "--")
table.style.hide(axis="index")

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.


Backbone Variant,Params (M),GFLOPs,Precision,Recall,F1-score,mAP@50,FPS (Rasp-5),FPS (Jetson)
Standard YOLO26n Backbone,2.380000,5.340000,0.971000,0.937000,0.953000,0.958000,13.380000,--
ADown Backbone (Paper 1),2.030000,4.490000,0.967000,0.925000,0.946000,0.952000,--,--


## 8. Zip all results

Run this last. It creates **`paper1_backbone_all_results.zip`** in the **Output** tab with:

| Folder in the zip | Contents |
|---|---|
| `paper1_backbone_edge_bench/` | NCNN + `.pt` model, 50 bench images, `pi5_bench.py`, `jetson_bench.py` |
| `weights/` | `best.pt`, renamed `bb_adown_best.pt` |
| `runs/` | full training folder: `best.pt`, `last.pt`, `results.csv`, training curves, confusion matrix, PR curves |
| `yolo26n-bb-adown.yaml` | the model definition |

In [10]:
import zipfile

ALL_ZIP = WORK / "paper1_backbone_all_results.zip"
folders = [WORK / "paper1_backbone_edge_bench", WORK / "runs"]

with zipfile.ZipFile(ALL_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in folders:
        if not folder.exists():
            print(f"{folder.name}: not found, skipped"); continue
        n = 0
        for f in folder.rglob("*"):
            if f.is_file():
                z.write(f, f.relative_to(WORK))
                n += 1
        print(f"{folder.name + '/':30s} {n} files")
    for _, run in VARIANTS:
        w = best_path(run)
        if w.exists():
            z.write(w, f"weights/{run}_best.pt")
            print(f"weights/{run}_best.pt")
    if Path(MODEL_YAML).exists():
        z.write(MODEL_YAML, Path(MODEL_YAML).name)

print(f"\nDownload {ALL_ZIP.name} ({ALL_ZIP.stat().st_size / 1e6:.1f} MB) from the Output tab.")

paper1_backbone_edge_bench/    58 files
runs/                          26 files
weights/bb_adown_best.pt

Download paper1_backbone_all_results.zip (39.4 MB) from the Output tab.
